# G2P PEFT — manual fallback notebook

Use this only if you prefer the Kaggle UI over the automated `scripts/03_push_and_poll.py` path (e.g. to use a **T4 x2** accelerator, which the API does not cleanly expose).

**Setup:**
1. Upload the `g2p-peft-bundle` dataset (built by `scripts/02_make_bundle.py`) and **Add Input** here. (If you have not built it, create a dataset whose single file is `kaggle/bundle/g2p_bundle.zip`.)
2. Settings: **Accelerator = GPU** (P100 or T4 x2), **Internet = ON**.
3. Set `LANG` below, **Run All**, then **Save Version → Save & Run All (Commit)**.
4. From the **Output** tab download `results_<lang>.json` and the `runs/` JSON files into your local `results/runs/`.
5. Repeat for the other language, then run `python build_artifacts.py` locally.

In [ ]:
import glob, os, subprocess, sys, zipfile

LANG = "rum"  # set to "rum" or "gre"; run the notebook once per language

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "peft==0.13.2"])

WORK = "/kaggle/working"
zips = glob.glob("/kaggle/input/*/g2p_bundle.zip")
if zips:
    bundle = os.path.join(WORK, "bundle")
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(bundle)
else:
    cands = glob.glob("/kaggle/input/*/g2p/__init__.py")
    assert cands, "g2p-peft-bundle not attached"
    bundle = os.path.dirname(os.path.dirname(cands[0]))

sys.path.insert(0, bundle)
data_dir = os.path.join(bundle, "data")
eng_test = os.path.join(data_dir, "eng_us_forgetting.tsv")
print("bundle:", bundle)
print("data files:", sorted(os.listdir(data_dir)))

from g2p.run_matrix import run_language
run_language(LANG, data_dir=data_dir, eng_test_path=eng_test, out_dir=WORK)
print("ALL DONE")